In [1]:
!pip install pyspark --quiet

In [2]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import expr

# Iniciar Sesion de PySpark


In [3]:
spark = SparkSession.builder \
    .appName("MovieLensALS") \
    .getOrCreate()

# Importar los 2 csv utilizados (ratings y movies)

In [5]:
ratings = spark.read.csv("ratings.csv", header=True, inferSchema=True)
ratings = ratings.select("userId", "movieId", "rating")

In [6]:
movies = spark.read.csv("movies.csv", header=True, inferSchema=True)

# Calculo del número de ratings totales de cada pelicula
Eliminamos las peliculas que tienen menos de 200 ratings, esto hace que las peliculas sean bastante más famosas, menos problemas al hacer las predicciones sin perder una gran cantidad de datos.

In [7]:
# 1. Calcular número de ratings por película
movie_counts = ratings.groupBy("movieId").count().withColumnRenamed("count", "rating_count")

# 2. Filtrar películas con al menos 10 ratings
popular_movies = movie_counts.filter(col("rating_count") >= 200)

# 3. Unir con el dataset original
ratings = ratings.join(popular_movies, on="movieId", how="inner")

De 1 millón de registros solo perdemos 150 mil a cambio de mejores predicciones

In [8]:
print(ratings.count())

855730


# Dividimos los datos en Train y Test, entrenamos el modelo de ALS

In [9]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

In [10]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    implicitPrefs=False,
    coldStartStrategy="drop"  # Evita NaN en predicciones
)

model = als.fit(train)

# Evaluamos los resultados del modelo

In [11]:
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse:.4f}")

RMSE: 0.8678


# Creamos un dataset que tenga todos los usuarios y todas las peliculas que cada usuario no haya puntuado

In [12]:
user_ids = ratings.select("userId").distinct()
movie_ids = ratings.select("movieId").distinct()
user_movie = user_ids.crossJoin(movie_ids)

# 7.1 Unir con ratings para detectar nulos y ORDENAR por userId y movieId

df_largo = user_movie.join(ratings, on=["userId", "movieId"], how="left") \
                     .orderBy("userId", "movieId")

# 7.2 Filtrar solo filas con rating nulo

df_nulos = df_largo.filter(col("rating").isNull())

# Predicción de los nulos en el dataset anterior

In [13]:
pred_nulos = model.transform(df_nulos)

pred_nulos.show(10, truncate=False)

+------+-------+------+------------+----------+
|userId|movieId|rating|rating_count|prediction|
+------+-------+------+------------+----------+
|1     |2      |NULL  |NULL        |3.30556   |
|1     |3      |NULL  |NULL        |3.1721017 |
|1     |7      |NULL  |NULL        |3.518371  |
|1     |11     |NULL  |NULL        |3.9792602 |
|1     |16     |NULL  |NULL        |3.5143275 |
|1     |19     |NULL  |NULL        |2.471808  |
|1     |22     |NULL  |NULL        |3.5780563 |
|1     |25     |NULL  |NULL        |3.2824845 |
|1     |32     |NULL  |NULL        |3.2287276 |
|1     |36     |NULL  |NULL        |3.8636236 |
+------+-------+------+------------+----------+
only showing top 10 rows



# Quitamos las columnas sobrantes en el dataset y añadimos una columna con el titulo de la pelicula.

In [14]:
pred_nulos.drop("rating").drop("rating_count").show(10, truncate=False)

+------+-------+----------+
|userId|movieId|prediction|
+------+-------+----------+
|1     |2      |3.30556   |
|1     |3      |3.1721017 |
|1     |7      |3.518371  |
|1     |11     |3.9792602 |
|1     |16     |3.5143275 |
|1     |19     |2.471808  |
|1     |22     |3.5780563 |
|1     |25     |3.2824845 |
|1     |32     |3.2287276 |
|1     |36     |3.8636236 |
+------+-------+----------+
only showing top 10 rows



In [15]:
df_final = pred_nulos.join(movies, on="movieId", how="left")

In [16]:
df_final.show(5)

+-------+------+------+------------+----------+--------------------+--------------------+
|movieId|userId|rating|rating_count|prediction|               title|              genres|
+-------+------+------+------------+----------+--------------------+--------------------+
|      2|     1|  NULL|        NULL|   3.30556|      Jumanji (1995)|Adventure|Childre...|
|      3|     1|  NULL|        NULL| 3.1721017|Grumpier Old Men ...|      Comedy|Romance|
|      7|     1|  NULL|        NULL|  3.518371|      Sabrina (1995)|      Comedy|Romance|
|     11|     1|  NULL|        NULL| 3.9792602|American Presiden...|Comedy|Drama|Romance|
|     16|     1|  NULL|        NULL| 3.5143275|       Casino (1995)|      Drama|Thriller|
+-------+------+------+------------+----------+--------------------+--------------------+
only showing top 5 rows



In [17]:
df_final = df_final.drop("rating").drop("rating_count").drop("genres")

Pasar el dataset a csv

In [18]:
#df_final.coalesce(1).write.csv("predicciones_peliculas_unico", header=True, mode="overwrite")